# 04 — LangGraph Pipeline
Walk through the full StateGraph: guardrails → supervisor → agents → HITL → synthesizer.

In [ ]:
import sys; sys.path.insert(0, '/home/claude/codebase/code/src')
import os; os.environ['ENABLE_MOCK']='true'; os.environ['REDIS_ENABLED']='false'

## Build and inspect the graph

In [ ]:
from graph.graph import get_graph, copilot_graph
from graph.state import initial_state

graph = get_graph()
print('Graph type:', type(graph))
print('Graph nodes:', list(graph.nodes.keys()) if hasattr(graph, 'nodes') else 'compiled')

## Run a full query

In [ ]:
state = initial_state(
    query='What is the current GRR for retention?',
    thread_id='notebook-demo-01',
    data_products=['retention']
)
result = copilot_graph.invoke(state, config={'configurable': {'thread_id': 'notebook-demo-01'}})

print('Intent:', result.get('intent'))
print('Summary:', result.get('final_summary'))
print('Confidence:', result.get('confidence'))
print('Agents ran:', result.get('_agents_ran'))
print('Anomalies:', result.get('anomalies'))
print('Execution ms:', result.get('execution_ms'))

## Multi-product query

In [ ]:
state2 = initial_state(
    query='Show me a full diagnostic on all our data products',
    thread_id='notebook-demo-02',
    data_products=['retention', 'bookings', 'ltv']
)
result2 = copilot_graph.invoke(state2, config={'configurable': {'thread_id': 'notebook-demo-02'}})
print('Agents ran:', result2.get('_agents_ran'))
print('\nSummary:', result2.get('final_summary'))

## Guardrail blocking

In [ ]:
blocked = initial_state(query='DROP TABLE retention_metrics', thread_id='blocked-01')
result3 = copilot_graph.invoke(blocked, config={'configurable': {'thread_id': 'blocked-01'}})
print('Guardrail passed:', result3.get('guardrail_passed'))
print('Summary:', result3.get('final_summary'))

## HITL (Human-in-the-Loop) flow

In [ ]:
# Step 1 — anomaly triggers pending_action
from services.databricks.mock import MockDatabricksService
from agents.information_agent import InformationAgent
from graph.graph import get_graph
import graph.nodes as n

# Inject low-grr service so anomaly is detected
n._agents['information'] = InformationAgent(data_service=MockDatabricksService(low_grr=True))

state_hitl = initial_state(
    query='Check retention metrics and create tickets if issues found',
    thread_id='hitl-demo-01',
    data_products=['retention']
)
graph = get_graph()
result_pending = graph.invoke(state_hitl, config={'configurable': {'thread_id': 'hitl-demo-01'}})
print('Pending action:', result_pending.get('pending_action'))
print('Auto tickets (before approval):', result_pending.get('auto_tickets'))

In [ ]:
# Step 2 — approve, tickets get created
state_approved = initial_state(
    query='Check retention metrics and create tickets if issues found',
    thread_id='hitl-demo-01',
    data_products=['retention'],
    approved=True,
    anomalies=result_pending.get('anomalies', ['GRR below threshold'])
)
result_approved = graph.invoke(state_approved, config={'configurable': {'thread_id': 'hitl-demo-01'}})
print('Auto tickets (after approval):', result_approved.get('auto_tickets'))
n._agents['information'] = InformationAgent()  # restore normal agent

## AgentState schema — all fields

In [ ]:
from graph.state import AgentState
from typing import get_type_hints
print('AgentState fields:')
for k, v in get_type_hints(AgentState).items():
    print(f'  {k}: {v}')